# A tiny language model

The deck ended on a network that computes the table of a language model. The corpus $x$ goes in, and out comes $\hat Y$: one row per row of $x$, one column per word of the dictionary, and in each cell the chance that the word of that column comes next. This notebook builds such a network for a language of six words and trains it in seconds.

* Every row reads a **window**: the last $C = 11$ words of its prefix. A shorter prefix is filled up in front with the word `PAD`.
* Every word gets **8 numbers**, its **embedding**, kept in a table. The table is a block of weights: training moves it, like every other weight of the network.
* The 11 embeddings of a window, side by side, go through a plain network of two layers: `nn.Linear`, a ReLU, `nn.Linear`. There is no attention: the window is all that a row reads.
* The output $\hat Y$ is a **matrix of chances**: every row is a text so far, every column a word, and every number the chance that this word comes next.

The names follow the deck:

| The deck | This notebook |
|---|---|
| the corpus, $x$ and $y$ | `corpus`, `x` and `y`, lists of words |
| the context $C$ | `C`, the length of a window |
| the embedding table | `word_table` |
| $E$ | `E` |
| $\hat Y$ | `Y_hat` |
| $\ell_i$ and $\bar\ell$ | `ell` and `ell_bar`, called `loss` in training |

Run the cells in order, or choose Runtime, Run all.

In [ ]:
#@title Imports { display-mode: "form" }
import math
import random
import torch
from torch import nn
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

## 1. Six words and three rules

The language has six words, `if`, `then`, `either`, `or`, `rain` and `sun`, and three rules:

* `rain` is a sentence, and so is `sun`.
* If A and B are sentences, then `if A then B` is a sentence.
* If A and B are sentences, then `either A or B` is a sentence.

So `if rain then sun` is a sentence, and so is `either sun or if rain then sun`. Every `then` must answer its own `if`, and every `or` its own `either`, however far back they are. In 1957 Noam Chomsky used exactly these two pairs to argue that no machine with a fixed, limited memory, such as one that remembers only the last few words, can produce English: sentences can nest inside each other to any depth.

`make_sentence` follows the three rules and picks at random. The deeper inside a sentence it is, the likelier it picks a plain `rain` or `sun`, so no sentence goes more than three levels deep. That limit is what lets this notebook's network, which remembers only the last 11 words, learn the language (section 3).

In [ ]:
random.seed(1)                                               # seeded: the same sentences on every computer
PLAIN = [0.1, 0.45, 0.7, 1.0]                                # the chance of a plain rain or sun at depth 0, 1, 2, 3


def make_sentence(depth=0):
    if random.random() < PLAIN[depth]:
        return [random.choice(["rain", "sun"])]                                          # rule 1
    if random.random() < 0.5:
        return ["if"] + make_sentence(depth + 1) + ["then"] + make_sentence(depth + 1)    # rule 2
    return ["either"] + make_sentence(depth + 1) + ["or"] + make_sentence(depth + 1)      # rule 3


for k in range(8):
    print(" ".join(make_sentence()))

In [ ]:
#@title `is_grammatical` checks a list of words against the three rules (double-click to view the code) { display-mode: "form" }
def read_one_sentence(words, i):
    # Reads one sentence that starts at word i. Returns the place where it ends, or -1 if a rule fails.
    if i >= len(words):
        return -1
    if words[i] in ("rain", "sun"):
        return i + 1
    if words[i] in ("if", "either"):
        middle = read_one_sentence(words, i + 1)
        partner = "then" if words[i] == "if" else "or"
        if middle == -1 or middle >= len(words) or words[middle] != partner:
            return -1
        return read_one_sentence(words, middle + 1)
    return -1


def is_grammatical(words):
    return read_one_sentence(words, 0) == len(words)


for text in ["if rain then sun", "either sun or if rain then sun", "if rain or sun", "if rain then"]:
    print(f"{text!r:36} grammatical: {is_grammatical(text.split())}")

## 2. The corpus, $x$ and $y$

The dictionary has seven words: the six of the language and `END`, the word that separates two texts. Here every text is one sentence of the language. As in the deck, the corpus reads 3000 texts one after the other into one column, with `END` before the first text, between every two texts and after the last.

* `x` is the corpus without its last `END`. Every word of `x` gets a row.
* `y` is the shifted corpus: the corpus without its first `END`, moved up one row. `y[i]` is the word that came after `x[i]`.

In [ ]:
WORDS = ["if", "then", "either", "or", "rain", "sun", "END"]   # the dictionary: one column of Y_hat per word

texts = [make_sentence() for k in range(3000)]               # 3000 texts of the language
corpus = ["END"]
for text in texts:
    corpus += text + ["END"]                                 # the words of the text, then END
x = corpus[:-1]                                              # the corpus without its last END
y = corpus[1:]                                               # the shifted corpus: y[i] is the word that came after x[i]
print(len(corpus), "words in the corpus, so x and y have", len(x), "rows each")

The first rows of `x` and `y`, side by side, with the **prefix** of every row. The prefix is read upward from the row to the first `END`, both included: it is the text so far. An `END` row is its own prefix: a text is about to start, and its `y` is the first word of that text. Python counts the rows from 0.

In [ ]:
top = 0                                                      # the END row that the current text starts at
print(" row  x[i]    y[i]    the prefix of row i")
for i in range(16):
    if x[i] == "END":
        top = i                                              # a new text starts at this row
    prefix = " ".join(x[top:i + 1])                          # read upward from row i to the END
    print(f"{i:>4}  {x[i]:<7} {y[i]:<7} {prefix}")

## 3. The window: the last 11 words of the prefix

A network has a fixed number of inputs, but prefixes come in every length. So every row reads a **window** of fixed length, the **context** $C$: the last $C$ words of its prefix, its own word last. A longer prefix is cut to its last $C$ words. A shorter one is filled up in front with `PAD`, a word that only fills the window and never comes next.

Here $C = 11$. Right after an inner sentence ends comes `then` or `or`, and only the word before that inner sentence, `if` or `either`, tells which. An inner sentence has at most 10 words, so the window must reach 11 words back. With 11 words a row always sees what the rules need to say which words may come next; with 10 it does not always.

In [ ]:
C = 11                                                       # the context: every row reads the last 11 words of its prefix


def window(prefix):
    return (["PAD"] * C + prefix)[-C:]                       # PAD in front, then keep the last C words


windows = []                                                 # the window of every row of x
for i in range(len(x)):
    if x[i] == "END":
        top = i                                              # a new text starts at this row
    windows.append(window(x[top:i + 1]))                     # its prefix, cut or filled up to C words

print(" row  x[i]    y[i]    the window of row i")
for i in range(16):
    print(f"{i:>4}  {x[i]:<7} {y[i]:<7} {' '.join(windows[i])}")

Every row is now one example on its own: a window of 11 words in, the word that came next out. The longest text of the corpus has 22 words, so the last rows of a long text see only the end of their prefix:

In [ ]:
longest = max(texts, key=len)
print(len(longest), "words:", " ".join(longest))
print("the window of its last word:", " ".join(window(["END"] + longest)))

## 4. The embeddings: one table

The network computes with numbers, so every word gets $d = 8$ numbers, its **embedding**, kept in a table. A window can hold 8 different words, the 7 of the dictionary and `PAD`, so the table is an `nn.Embedding(8, 8)`: a row of 8 numbers for each of them. Given a block of word numbers, it returns the row of each. The table starts as small random numbers, with a spread of $0.02$, so every word starts close to $0$, and it is a block of weights: training moves it.

A word is known by its place in `TOKENS`, the dictionary followed by `PAD`: `if` is word 0, `then` word 1, and so on up to `END`, word 6, and `PAD`, word 7. `numbers` turns a list of words into these numbers. `X` holds the windows of all rows as numbers, one row per row of `x` and $C = 11$ columns, and `Y` holds the words that came next.

In [ ]:
TOKENS = WORDS + ["PAD"]                                     # the words a window can hold: the dictionary, then PAD


def numbers(words):
    return [TOKENS.index(word) for word in words]            # every word by its place in TOKENS


X = torch.tensor([numbers(words) for words in windows])      # the windows as numbers: one row per row of x, C columns
Y = torch.tensor(numbers(y))                                 # the words that came next, as numbers
print("X:", tuple(X.shape), "  Y:", tuple(Y.shape))
print("the window of row 2 as numbers:", X[2].tolist())

In [ ]:
torch.manual_seed(1)                                         # seeded: the same starting numbers on every computer
d = 8                                                        # numbers per word

word_table = nn.Embedding(len(TOKENS), d)                    # 8 words, 8 numbers each
nn.init.normal_(word_table.weight, std=0.02)                 # every word starts close to 0
word_table_before = word_table.weight.detach().clone()       # kept, to see later how far each word moved
print("word table:", tuple(word_table.weight.shape))

In [ ]:
#@title The word table as a picture (double-click to view the code) { display-mode: "form" }
BLUE_WHITE_RED = LinearSegmentedColormap.from_list("blue_white_vermillion", ["#0072B2", "#FFFFFF", "#D55E00"])
BLUES = LinearSegmentedColormap.from_list("white_blue", ["#FFFFFF", "#0072B2"])
CIRCLE = "#D55E00"


def show_tables(tables, titles):
    fig, axes = plt.subplots(1, len(tables), figsize=(4.8 * len(tables), 3.2), squeeze=False,
                             gridspec_kw={"wspace": 0.4})
    for ax, table, title in zip(axes[0], tables, titles):
        picture = ax.imshow(table.detach().numpy(), cmap=BLUE_WHITE_RED, vmin=-1.5, vmax=1.5, aspect="auto")
        ax.set_yticks(range(len(TOKENS)), TOKENS)
        ax.set_xticks(range(d), [str(k + 1) for k in range(d)])
        ax.set_xlabel("the 8 numbers of each word")
        ax.set_title(title, loc="left")
    fig.colorbar(picture, ax=list(axes[0]), shrink=0.85)
    plt.show()


show_tables([word_table_before], ["The word table before training"])

Before training the picture is blank: every number is close to $0$, and no word is like any other yet.

## 5. The network: a window in, seven scores out

`word_table(X)` looks up the 8 numbers of every word of every window, and `reshape` puts the 11 rows of a window side by side: row $i$ of `E` holds $11 \times 8 = 88$ numbers, the embedding of the first word of its window, then that of the second, and so on. Every word has its own slot, so the network can tell `if rain then` from `then rain if`.

A plain network of two layers then turns every row of `E` into 7 scores, one for each word that could come next: an `nn.Linear(88, 32)`, a ReLU, and an `nn.Linear(32, 7)`. An `nn.Linear` is a matrix with its biases, which training moves. There is no attention: a row reads its window and nothing else.

In [ ]:
network = nn.Sequential(
    nn.Linear(C * d, 32), nn.ReLU(),                         # in: the 88 numbers of a window
    nn.Linear(32, len(WORDS)),                               # out: 7 scores, one per word of the dictionary
)


def next_word_scores(X):                                     # X: windows as numbers, one per row
    E = word_table(X).reshape(len(X), C * d)                 # row i: the 11 embeddings of its window, side by side
    return network(E)                                        # one row of 7 scores per window

## 6. The output: the matrix $\hat Y$

The softmax turns every row of 7 scores into 7 chances that add up to $1$: it takes $e$ to the power of every score and divides by their sum. **Every row is the text so far, every column is a word, and every number is the chance that this word comes next.** `windows_of` gives the windows of the rows of one text, `END` and its words.

In [ ]:
def chances(X):
    return next_word_scores(X).softmax(dim=1)                # one row per window, 7 chances each


def windows_of(rows):                                        # the rows of one text: END and its words
    return torch.tensor([numbers(window(rows[:i + 1])) for i in range(len(rows))])


example = "if either rain or sun then sun".split()
x_example = ["END"] + example                                # its rows of x
y_example = example + ["END"]                                # its rows of y
Y_hat_example = chances(windows_of(x_example))               # its rows of Y_hat
print(tuple(Y_hat_example.shape), "  8 rows, one per row of x; 7 columns, one per word")
print("every row adds up to", Y_hat_example.sum(dim=1).detach())

In [ ]:
#@title The matrix of chances as a table, the word that came next circled (double-click to view the code) { display-mode: "form" }
def show_chances(x_rows, y_rows, title):
    with torch.no_grad():
        table = chances(windows_of(x_rows)).numpy()
    prefixes = [" ".join(x_rows[:i + 1]) for i in range(len(x_rows))]
    fig, ax = plt.subplots(figsize=(9.4, 0.42 * len(prefixes) + 1.3))
    ax.imshow(table, cmap=BLUES, vmin=0, vmax=1, aspect="auto")
    for row in range(table.shape[0]):
        for column in range(table.shape[1]):
            ax.text(column, row, f"{table[row, column]:.2f}", ha="center", va="center", fontsize=10,
                    color="white" if table[row, column] > 0.6 else "black")
        ax.plot(WORDS.index(y_rows[row]), row, "o", markersize=27, markerfacecolor="none",
                markeredgecolor=CIRCLE, markeredgewidth=2)   # the circle: y_i, the word that came next
    ax.set_xticks(range(len(WORDS)), WORDS)
    ax.xaxis.tick_top()
    ax.set_yticks(range(len(prefixes)), prefixes)
    ax.set_ylabel("the prefix of the row")
    ax.set_title(title, loc="left", pad=26)
    plt.show()


show_chances(x_example, y_example, "Before training: the chance of every word, after every prefix")

Before training every chance is close to $1/7 = 0.14$: the model has no idea yet which word comes next. The circle in every row marks the entry of the word that came next, $\hat y_{i, y_i}$, the entry the loss reads.

A row reads only its window, so one call computes the rows of all rows of `x` at once, the whole matrix $\hat Y$:

In [ ]:
with torch.no_grad():
    scores = next_word_scores(X)                             # every row of x, 7 scores each, before training
Y_hat = scores.softmax(dim=1)                                # the whole matrix of chances
print("Y_hat:", tuple(Y_hat.shape), "  one row per row of x, one column per word")
print(f"every row adds up to 1: the sums lie between {Y_hat.sum(dim=1).min():.6f} and {Y_hat.sum(dim=1).max():.6f}")

## 7. The loss, by hand

The loss of the deck. In every row, take the entry of the word that came next, $\hat y_{i, y_i}$, the circled one, and its minus logarithm:

$$\ell_i = -\log \hat y_{i, y_i}.$$

The loss $\bar\ell$ is the average of the column $\ell$. A row that gave the word that came next a chance of $1$ costs $-\log 1 = 0$; the smaller that chance, the more the row costs. `loss_by_hand` does this for the rows of one text, and prints the column.

In [ ]:
def loss_by_hand(x_rows, y_rows):
    Y_hat_rows = chances(windows_of(x_rows))
    ell = []                                                 # the loss column, one entry per row
    for i in range(len(x_rows)):
        circled = Y_hat_rows[i, WORDS.index(y_rows[i])].item()   # the chance row i gave the word that came next
        ell.append(math.log(1 / circled))                    # ell_i = -log yhat_{i, y_i} = log(1 / yhat_{i, y_i})
        prefix = " ".join(x_rows[:i + 1])
        print(f"{i:>3}  {prefix:<34} y_i = {y_rows[i]:<6}  yhat = {circled:.3f}   ell_i = {ell[i]:.3f}")
    return sum(ell) / len(ell)                               # ell_bar: the average of the column


ell_bar = loss_by_hand(x_example, y_example)
print()
print(f"ell_bar by hand:              {ell_bar:.4f}")
print(f"nn.functional.cross_entropy:  {nn.functional.cross_entropy(next_word_scores(windows_of(x_example)), torch.tensor(numbers(y_example))).item():.4f}")

`nn.functional.cross_entropy` takes the scores before the softmax and the words that came next, and does the softmax, the minus logarithm and the average itself: it gives the same number. Before training every $\ell_i$ is close to $\log 7 = 1.95$, and so is $\bar\ell$ on the whole corpus:

In [ ]:
print(f"the loss on the whole corpus: {nn.functional.cross_entropy(scores, Y).item():.4f}")

## 8. Training

Training makes $\bar\ell$ small, 256 rows at a time.

* A step picks 256 rows of `x` at random: their windows from `X`, and the words that came next from `Y`.
* `nn.functional.cross_entropy` gives $\bar\ell$ of these rows, the `loss`.
* `loss.backward()` computes the gradient of the loss for every weight, and `optimizer.step()` moves them all: the weights of the network, and **also the 8 numbers of every word**, since the optimizer lists the word table beside the network. Adam is gradient descent with a step size that it adapts for every weight.
* The loop takes 500 steps, so it picks every row of `x` about 4.5 times.

The loss cannot reach $0$. Wherever a new sentence can start, at the top of a text and right after `if`, `then`, `either` or `or`, the rules allow `if`, `either`, `rain` and `sun` (only the last two at the deepest level), and chance decides between them. Even a model that knew `make_sentence` by heart would score about $0.73$.

In [ ]:
optimizer = torch.optim.Adam(list(word_table.parameters()) + list(network.parameters()), lr=0.01)

losses = []                                                  # the loss of every step, kept for the plot below
for step in range(500):
    pick = torch.randint(0, len(x), (256,))                  # 256 rows of x, picked at random
    loss = nn.functional.cross_entropy(next_word_scores(X[pick]), Y[pick])   # ell_bar of their rows
    optimizer.zero_grad()                                    # forget the gradients of the previous step
    loss.backward()                                          # compute the gradient of the loss
    optimizer.step()                                         # move every weight, and every word's 8 numbers
    losses.append(loss.item())
    if (step + 1) % 100 == 0:
        print(f"step {step + 1}: the loss of its 256 rows is {loss.item():.4f}")

The loss of every step, and its running average, which keeps 0.9 of itself and adds 0.1 of each new loss. The loss of 256 rows jumps around from step to step; its running average falls fast, then flattens out close to the floor:

In [ ]:
#@title The loss during training (double-click to view the code) { display-mode: "form" }
running = []                                                 # the running average after every step
average = losses[0]
for loss_of_step in losses:
    average = 0.9 * average + 0.1 * loss_of_step             # keep 0.9 of the average, add 0.1 of the new loss
    running.append(average)

steps = range(1, len(losses) + 1)
fig, ax = plt.subplots(figsize=(8.4, 3.4))
ax.plot(steps, losses, color="#AAAAAA", linewidth=0.8, label="the loss of the step")
ax.plot(steps, running, color="#0072B2", linewidth=2, label="its running average")
ax.axhline(0.73, color="#D55E00", linestyle="--", linewidth=1, label="the floor, about 0.73")
ax.set_xlabel("step")
ax.set_ylabel("the loss")
ax.set_ylim(0, 2.2)
ax.legend()
plt.show()

## 9. The matrix after training

The same example, and its loss column:

In [ ]:
show_chances(x_example, y_example, "After training: the chance of every word, after every prefix")

In [ ]:
print(f"\nell_bar of the example: {loss_by_hand(x_example, y_example):.4f}")

What to look for:

* Where the rules leave a choice, the model spreads its chances over the words the rules allow: at the top of a text, after the prefix `END`, a sentence can open with any of `if`, `either`, `rain` and `sun`, and after `END if` as well. These rows keep a loss above $0$.
* It even learns how often each choice comes: `make_sentence` opens a text with `if` or `either` 45 times in 100 each, and with `rain` or `sun` 5 times each, and the first row reads close to that.
* Where the rules force a word, the model puts nearly everything on it: after `END if either rain` only `or` can come, and after `END if either rain or sun` only `then`, the answer to the `if` four words back. After the last `sun` the text is complete, and `END` must come. These rows cost almost nothing.
* The words the rules forbid get almost nothing.

On the whole corpus, the loss ends close to the floor of $0.73$:

In [ ]:
with torch.no_grad():
    scores = next_word_scores(X)                             # every row of x, 7 scores each, after training
Y_hat = scores.softmax(dim=1)
print("Y_hat:", tuple(Y_hat.shape))
print(f"the loss on the whole corpus: {nn.functional.cross_entropy(scores, Y).item():.4f}")

## 10. The word table after training

In [ ]:
#@title Before and after, and every word's nearest word (double-click to view the code) { display-mode: "form" }
show_tables([word_table_before, word_table.weight], ["Before training", "After training"])
after = word_table.weight.detach()
for word in WORDS:
    distances = [(after[TOKENS.index(word)] - after[TOKENS.index(other)]).norm().item() for other in WORDS]
    others = [(distance, other) for distance, other in zip(distances, WORDS) if other != word]
    distance, nearest = min(others)
    print(f"{word:>7}: nearest word {nearest!r:9} at a distance of {distance:.2f}")

Nobody told the model which words are alike. Two pairs end up nearest: `rain` and `sun`, which the rules use in the same way, and `then` and `or`, after each of which a second sentence starts in the same way. The other words keep further apart, because the model needs to tell them apart: `if` and `either` both open a sentence, but they wait for different partners.

## 11. The model writes

`write_text` starts from the prefix `END`, a text about to start. It takes the window of the text so far, gets its 7 chances, picks the next word with those chances, like a spin of the deck's roulette, and goes on until it picks `END`.

In [ ]:
def write_text():
    prefix = ["END"]                                         # the prefix END: a text is about to start
    next_word = ""
    while next_word != "END" and len(prefix) < 40:
        with torch.no_grad():
            row = chances(torch.tensor([numbers(window(prefix))]))[0]   # the chances after the window of the text so far
        next_word = random.choices(WORDS, weights=row.tolist())[0]     # one word, picked with these 7 chances
        prefix.append(next_word)
    return [word for word in prefix if word != "END"]        # the text, without its ENDs


for k in range(10):
    text = write_text()
    print("yes" if is_grammatical(text) else "NO ", " ".join(text))

In [ ]:
made = [write_text() for k in range(1000)]
grammatical = [text for text in made if is_grammatical(text)]
new = [text for text in grammatical if text not in texts]
print(len(grammatical), "of 1000 texts follow the rules, and", len(new), "of them are not among the 3000 it learned from")

## 12. Try it

* Set `C = 10` in section 3 and choose Runtime, Run all. With one word less, the window misses the `if` or `either` before the longest inner sentences. In our runs 87 to 96 texts in 100 then followed the rules, against 98 to 99 with `C = 11`; with `C = 5`, 58 to 73.
* Change `d`, the 32 numbers of the hidden layer or the 500 steps, and watch the loss and the texts.